# FlagOS Track 3: LLM Automatic Data Annotation in Long-Context Scenarios
## ICL-Based Solution with Qwen3-4B + Self-Consistency

**Team**: zan-maker  
**Approach**: Multi-strategy In-Context Learning (ICL) with chain-of-thought reasoning, self-consistency decoding, and long-context window optimization using Qwen3-4B.

In [ ]:
# ============================================================
# INSTALLATION
# ============================================================
!pip install -q transformers accelerate bitsandbytes
!pip install -q sentencepiece protobuf
!pip install -q scipy scikit-learn

In [ ]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
import os
import json
import re
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from collections import Counter

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================
# HYPERPARAMETERS
# ============================================================
MODEL_NAME = "Qwen/Qwen3-4B"
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.3
TOP_P = 0.9
NUM_SHOT_EXAMPLES = 5
SELF_CONSISTENCY_RUNS = 3
USE_4BIT = True
CONTEXT_MAX_CHARS = 18000  # Conservative for 32K token limit

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# ============================================================
# LOAD & EXPLORE DATA
# ============================================================
INPUT_DIR = "/kaggle/input/track-3-llm-automatic-data-annotation-in-long-context-scenarios"

print("=== Data Files ===")
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        print(f"  {f} ({os.path.getsize(fpath)/1024:.1f} KB)")

# Load data (adapt column names to actual competition format)
data_files = {}
for fname in os.listdir(INPUT_DIR):
    fpath = os.path.join(INPUT_DIR, fname)
    if fname.endswith('.csv'):
        data_files[fname] = pd.read_csv(fpath)
    elif fname.endswith('.json'):
        data_files[fname] = pd.read_json(fpath)
    elif fname.endswith('.parquet'):
        data_files[fname] = pd.read_parquet(fpath)

print("\n=== Loaded Files ===")
for k, v in data_files.items():
    print(f"\n{k}: shape={v.shape}, columns={list(v.columns)}")
    print(v.head(2).to_string())

In [ ]:
# ============================================================
# PREPARE DATA
# ============================================================
# Identify train/test/sample_submission files
train_df = pd.DataFrame()
test_df = pd.DataFrame()
sample_sub = pd.DataFrame()

for k, v in data_files.items():
    kl = k.lower()
    if 'train' in kl:
        train_df = v
    elif 'test' in kl:
        test_df = v
    elif 'sample' in kl:
        sample_sub = v

# If only test and sample_submission found
if test_df.empty and not sample_sub.empty:
    test_df = sample_sub.drop(columns=[c for c in sample_sub.columns if c.lower() in ['predicted', 'prediction', 'label', 'annotation']]) if len(sample_sub.columns) > 1 else sample_sub

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")
print(f"Sample Sub: {sample_sub.shape}")

# Extract label schema
label_schema = []
label_col = None
for col in ['label', 'annotation', 'category', 'class', 'target']:
    if col in train_df.columns:
        label_col = col
        label_schema = train_df[col].unique().tolist()
        break

if not label_schema:
    label_schema = ['positive', 'negative', 'neutral']  # Default fallback

print(f"\nLabel schema: {label_schema}")
print(f"Label column: {label_col}")

In [ ]:
# ============================================================
# LOAD MODEL
# ============================================================
print(f"Loading {MODEL_NAME}...")

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {"trust_remote_code": True, "torch_dtype": torch.float16, "device_map": "auto"}
if bnb_config:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.eval()

print(f"Model loaded! Device: {model.device}")
if torch.cuda.is_available():
    print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ============================================================
# ICL PROMPT BUILDER
# ============================================================
def build_icl_prompt(context, query, label_schema, examples=None, cot=True):
    """Build ICL prompt for annotation."""
    system_msg = (
        "You are an expert data annotator for long-context understanding. "
        "Annotate the target text based on the provided context and label schema. "
        "Think step-by-step for accuracy."
    )

    schema_str = ", ".join(label_schema)
    parts = [f"<|im_start|>system\n{system_msg}<|im_end|>"]
    parts.append(f"<|im_start|>user\nLabel Schema: [{schema_str}]\n")

    # Few-shot examples
    if examples:
        parts.append("Annotated Examples:\n")
        for i, ex in enumerate(examples, 1):
            parts.append(f"\nExample {i}:\nContext: {ex['context'][:1500]}\nTarget: {ex['target'][:500]}\n")
            if cot:
                parts.append(f"Reasoning: {ex['reasoning']}\n")
            parts.append(f"Annotation: {ex['label']}\n")

    # Task
    ctx = str(context)[:CONTEXT_MAX_CHARS]
    parts.append(f"\nNow annotate this:\nContext: {ctx}\nQuery: {str(query)[:1000]}\n")
    if cot:
        parts.append("Reason step by step, then give the final annotation.\n")
    parts.append("<|im_end|>")
    parts.append("<|im_start|>assistant")

    return "\n".join(parts)


def generate_response(prompt, temperature=TEMPERATURE):
    """Generate model response."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=32768).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=temperature,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def extract_label(response, label_schema):
    """Extract annotation label from model response."""
    resp_lower = response.lower()
    for label in label_schema:
        if label.lower() in resp_lower:
            return label
    # Try patterns
    for pat in [r"annotation:\s*(.+?)(?:\n|$)", r"answer:\s*(.+?)(?:\n|$)"]:
        m = re.search(pat, resp_lower)
        if m:
            for label in label_schema:
                if label.lower() in m.group(1):
                    return label
    return label_schema[0]  # Fallback


def self_consistency_predict(prompt, label_schema, runs=3):
    """Multi-run self-consistency."""
    votes = Counter()
    for i in range(runs):
        temp = TEMPERATURE + i * 0.15
        resp = generate_response(prompt, temperature=temp)
        label = extract_label(resp, label_schema)
        votes[label] += 1
    return votes.most_common(1)[0][0]

In [ ]:
# ============================================================
# PREPARE ICL EXAMPLES FROM TRAINING DATA
# ============================================================
icl_examples = []
if not train_df.empty and label_col:
    ctx_col = next((c for c in train_df.columns if c in ['context','documents','text','input','passage']), None)
    q_col = next((c for c in train_df.columns if c in ['query','question','target_text','target']), None)

    if ctx_col:
        for lbl in train_df[label_col].unique():
            subset = train_df[train_df[label_col] == lbl]
            sampled = subset.sample(n=min(2, len(subset)), random_state=42)
            for _, row in sampled.iterrows():
                icl_examples.append({
                    "context": str(row[ctx_col])[:2000],
                    "target": str(row[q_col])[:500] if q_col else "",
                    "label": str(row[label_col]),
                    "reasoning": f"Based on the context indicators, the annotation is '{row[label_col]}'.",
                })

random.shuffle(icl_examples)
icl_examples = icl_examples[:NUM_SHOT_EXAMPLES]
print(f"ICL examples prepared: {len(icl_examples)}")

In [ ]:
# ============================================================
# GENERATE PREDICTIONS
# ============================================================
# Identify columns
id_col = next((c for c in test_df.columns if c.lower() in ['id', 'sample_id']), test_df.columns[0])
ctx_col = next((c for c in test_df.columns if c in ['context','documents','text','input','passage']), None)
q_col = next((c for c in test_df.columns if c in ['query','question','target_text','target']), None)

print(f"ID column: {id_col}")
print(f"Context column: {ctx_col}")
print(f"Query column: {q_col}")

predictions = []
total = len(test_df)

for idx, row in test_df.iterrows():
    sample_id = row[id_col]
    context = str(row[ctx_col]) if ctx_col else str(row.iloc[1])
    query = str(row[q_col]) if q_col else ""

    print(f"\n[{idx+1}/{total}] ID={sample_id}")
    t0 = time.time()

    try:
        prompt = build_icl_prompt(context, query, label_schema, icl_examples)
        pred = self_consistency_predict(prompt, label_schema, SELF_CONSISTENCY_RUNS)
    except Exception as e:
        print(f"  Error: {e}")
        pred = label_schema[0]

    elapsed = time.time() - t0
    print(f"  -> {pred} ({elapsed:.1f}s)")
    predictions.append({"ID": sample_id, "Predicted": pred})

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# SAVE SUBMISSION
# ============================================================
sub_df = pd.DataFrame(predictions)

# Align with sample_submission format
if not sample_sub.empty and 'ID' in sample_sub.columns:
    sub_df = sub_df.set_index('ID').reindex(sample_sub['ID']).reset_index()

submission_path = "/kaggle/working/submission.csv"
sub_df.to_csv(submission_path, index=False)

print(f"\n=== SUBMISSION SAVED ===")
print(f"Path: {submission_path}")
print(f"Total: {len(sub_df)} predictions")
print(f"\nPreview:")
print(sub_df.head(10))
print(f"\nLabel Distribution:")
print(sub_df['Predicted'].value_counts())